# 🧪 AcademiaSD LoRAlab Trainer Studio — Free Colab Edition

Train **image / video LoRAs for free** on a Colab **Tesla T4** (15 GB VRAM · 12 GB RAM · ~70 GB disk) using
[AcademiaSD/AcademiaSD_LoRAlab-TrainerStudio](https://github.com/AcademiaSD/AcademiaSD_LoRAlab-TrainerStudio).

### ▶️ How to use
1. **Runtime ▸ Change runtime type ▸ T4 GPU** (free).
2. Pick your trainer in **cell 1** (optional: enable Drive, tokens, password).
3. **Runtime ▸ Run all** — or run cells 1 → 5 one by one. First install takes ~6-10 min.
4. The last cell prints a **public link** (Cloudflare + fallback). Open it, log in, upload your dataset, caption, pre-cache, train.
5. **Leave the last cell running** — it is the keep-alive monitor. Stopping it only closes the tunnels; training keeps going and re-running the cell reconnects.

### 🧭 Cells
| # | Cell | What it does |
|---|------|--------------|
| 1 | ⚙️ Settings | all options in one form |
| 2 | 🧰 Toolkit | checks GPU / RAM / disk, defines helpers |
| 3 | 💾 Drive | mounts Google Drive **only if enabled** |
| 4 | 📦 Install | repo + Python 3.13 venv + PyTorch + dependencies |
| 5 | 🚀 Launch | trainer + 2 tunnels + keep-alive + live status |
| ➕ | 📜 Logs · 📤 Export · 💽 Disk · ⏹️ Stop | utilities |

> ⚠️ **T4 notes:** no native bf16 (Turing), so steps are slower than on newer GPUs. Start at 512 px, rank 8-16, batch 1. ✅ models in the dropdown fit comfortably; ⚠️ ones are tight.


In [ ]:
#@title ⚙️ 1 · Settings — pick your trainer, then run { display-mode: "form" }

#@markdown ## 🧠 Trainer
#@markdown ✅ fits a T4 comfortably &nbsp;·&nbsp; ⚠️ tight (use 512 px / low rank) &nbsp;·&nbsp; 🧪 experimental
TRAINER = "✅ Krea 2 · image · 8 GB [krea2]" #@param ["✅ SDXL · Pony / Illustrious / NoobAI / Juggernaut · 4 GB [sdxl]", "✅ Anima · anime & illustration · 4 GB [anima]", "✅ Z-Image / Z-Image-Turbo · image · 8 GB [zimage]", "✅ ERNIE-Image · image · under 8 GB [ernie]", "✅ MOSS-Image · image · 6 GB [mossimage]", "✅ Qwen-Image 2512 / Edit 2511 · image & edit · 4-8 GB [qwenimage2512]", "✅ FLUX.3 Image Klein · image & edit · 8 GB [flux3klein]", "✅ Krea 2 · image · 8 GB [krea2]", "✅ Qwen-Image 2.1 · image & edit · 8 GB [qwenimage21]", "✅ Ming-Image · transparent designs · 7 GB [mingimage]", "⚠️ FLUX.1 Dev · image · ~10 GB [flux1]", "⚠️ Boogu Image Edit · edit · 8-10 GB [boogu]", "⚠️ Fibo 2.0 (experimental) · 10 GB [fibo2]", "⚠️ MiniMax-H3 · video & audio · 8 GB+ [minimaxh3]", "⚠️ FLUX.2 Klein 9B · image & edit · 12 GB (tight) [klein9b]", "⚠️ Ideogram 4 · image · 12 GB (tight) [ideogram4]", "⚠️ Wan 2.2 T2V-A14B · video · 10.4 GB at 448px (tight) [wan22]", "⚠️ LTX 2.3 · video · 12 GB (tight) [ltx23]", "🧪 YuE2-3B · music (beta) · ~8 GB [yue2]"]

#@markdown ---
#@markdown ## 💾 Google Drive (optional)
#@markdown Off = everything lives on the temporary VM (use the 📤 Export cell or the in-app *Download LoRA* button before the session ends).
MOUNT_DRIVE = False #@param {type:"boolean"}
DRIVE_FOLDER = "LoRAlab" #@param {type:"string"}
PERSIST = "Light: Outputs + settings + dataset (recommended)" #@param ["Light: Outputs + settings + dataset (recommended)", "Medium: Light + Cache_Data (keeps pre-cache)", "Full: Medium + Models (needs a lot of free Drive space)"]

#@markdown ---
#@markdown ## 🌐 Public tunnels
#@markdown *Fallback* = **ngrok** if you give a token (best performance, free token at dashboard.ngrok.com), otherwise **localhost.run**, then **Pinggy** (no signup).
TUNNELS = "Both: Cloudflare + fallback" #@param ["Both: Cloudflare + fallback", "Cloudflare only", "Fallback only"]
NGROK_TOKEN = "2gpD2o8h60lirMzKYVbAHIWCcHT_4r2sFrqukbJ52kuqU9sdT" #@param {type:"string"}
#@markdown *(Tip: tokens can also be stored in Colab ▸ 🔑 Secrets as `NGROK_AUTHTOKEN` and `HF_TOKEN`; they are picked up automatically.)*

#@markdown ---
#@markdown ## 🔒 Access protection
#@markdown The trainer can launch processes and write files, so keep this ON. Leave the password empty to auto-generate one.
REQUIRE_PASSWORD = False #@param {type:"boolean"}
WEB_USER = "loralab" #@param {type:"string"}
WEB_PASSWORD = "" #@param {type:"string"}

#@markdown ---
#@markdown ## 🤗 Hugging Face token (optional)
#@markdown Most trainers download ungated NF4 models and need no token.
HF_TOKEN = "" #@param {type:"string"}

#@markdown ---
#@markdown ## 🛡️ Keep-alive & stability
KEEPALIVE_JS = True #@param {type:"boolean"}
KEEPALIVE_AUDIO = True #@param {type:"boolean"}
AUTO_RESTART_SERVER = True #@param {type:"boolean"}
#@markdown Swap file (GB) as a safety net for the 12 GB RAM limit. Best effort: Colab sometimes blocks swap, in which case it is skipped. `0` = off.
SWAP_GB = 8 #@param {type:"slider", min:0, max:16, step:2}

#@markdown ---
#@markdown ## 🔧 Advanced
REPO_REF = "main" #@param {type:"string"}

print("✅ Settings stored. Now run cell 2 (or Runtime ▸ Run all).")


✅ Settings stored. Now run cell 2 (or Runtime ▸ Run all).


In [ ]:
#@title 🧰 2 · Load toolkit & check this runtime { display-mode: "form" }
import os, sys, re, json, time, shutil, signal, subprocess, secrets, socket, base64, io, wave, struct, math, hashlib
import urllib.request, urllib.error
from pathlib import Path
from IPython.display import display, HTML, Javascript

# ─────────────────────────────────────────────────────────────────────────────
# Friendly errors: every expected problem = "what happened" + "how to fix it".
# ─────────────────────────────────────────────────────────────────────────────
class LoRALabError(Exception):
    pass

def die(what, fix):
    raise LoRALabError(what + "\n\n👉 HOW TO FIX: " + fix)

def _show_error(shell, etype, value, tb, tb_offset=None):
    bar = "─" * 78
    print("\n" + bar + "\n❌ " + str(value) + "\n" + bar)
    print("📁 Logs live in /content/loralab_logs/  (run the '📜 Show logs' cell to read them)")

try:
    get_ipython().set_custom_exc((LoRALabError,), _show_error)
except Exception:
    pass

# ─────────────────────────────────────────────────────────────────────────────
# Settings sanity check
# ─────────────────────────────────────────────────────────────────────────────
_needed = ["TRAINER", "MOUNT_DRIVE", "DRIVE_FOLDER", "PERSIST", "TUNNELS", "NGROK_TOKEN", "REQUIRE_PASSWORD",
           "WEB_USER", "WEB_PASSWORD", "HF_TOKEN", "KEEPALIVE_JS", "KEEPALIVE_AUDIO", "AUTO_RESTART_SERVER",
           "SWAP_GB", "REPO_REF"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    die("Settings are missing: " + ", ".join(_missing), "Run cell 1 (⚙️ Settings) first, or use Runtime ▸ Run all.")

TRAINER_IDS = {"sdxl", "anima", "zimage", "ernie", "mossimage", "qwenimage2512", "flux3klein", "krea2", "qwenimage21",
               "mingimage", "flux1", "boogu", "fibo2", "minimaxh3", "klein9b", "ideogram4", "wan22", "ltx23", "yue2"}
_m = re.search(r"\[([a-z0-9]+)\]\s*$", str(TRAINER).strip())
TRAINER_ID = _m.group(1) if _m else str(TRAINER).strip().lower()
if TRAINER_ID not in TRAINER_IDS:
    die("Unknown trainer: " + str(TRAINER), "Pick one from the dropdown in cell 1 (the id is the part in [brackets]).")

# ─────────────────────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────────────────────
ROOT = Path("/content")
if not ROOT.exists() or not os.access(str(ROOT), os.W_OK):
    die("This notebook is built for Google Colab (/content is not writable here).",
        "Open it in https://colab.research.google.com and choose Runtime ▸ Change runtime type ▸ T4 GPU.")
APP = ROOT / "AcademiaSD_LoRAlab-TrainerStudio"
VENV = ROOT / "loralab_venv"
PY = VENV / "bin" / "python"
LOGS = ROOT / "loralab_logs"
LOGS.mkdir(exist_ok=True)
PORT = 5000
REPO_URL = "https://github.com/AcademiaSD/AcademiaSD_LoRAlab-TrainerStudio.git"
WRAPPER = ROOT / "loralab_run.py"
LAST_LOG = None
OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))

# ─────────────────────────────────────────────────────────────────────────────
# Small helpers
# ─────────────────────────────────────────────────────────────────────────────
def log_tail(path, n=30):
    try:
        txt = Path(path).read_text(errors="ignore")
        txt = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", txt).replace("\r", "\n")
        lines = [l for l in txt.splitlines() if l.strip()]
        return "\n".join(lines[-n:]) if lines else "(log is empty)"
    except Exception:
        return "(no log yet)"

def indent(txt, pad="      "):
    return "\n".join(pad + l for l in str(txt).splitlines())

def hms(sec):
    sec = int(sec)
    return "%d:%02d:%02d" % (sec // 3600, sec % 3600 // 60, sec % 60)

def get_secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

def port_open(port=PORT, host="127.0.0.1", timeout=0.6):
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(timeout)
            return s.connect_ex((host, int(port))) == 0
    except Exception:
        return False

def http_get(url, timeout=10, headers=None):
    h = {"User-Agent": "Mozilla/5.0 (LoRAlab-Colab)", "ngrok-skip-browser-warning": "1"}
    if headers:
        h.update(headers)
    req = urllib.request.Request(url, headers=h)
    try:
        with OPENER.open(req, timeout=timeout) as r:
            return r.status, r.read(200000)
    except urllib.error.HTTPError as e:
        try:
            body = e.read(2000)
        except Exception:
            body = b""
        return e.code, body
    except Exception as e:
        return None, str(e).encode()

def _kill(proc, sig=signal.SIGTERM):
    if proc is None:
        return
    try:
        os.killpg(os.getpgid(proc.pid), sig)
    except Exception:
        try:
            proc.terminate()
        except Exception:
            pass

def sh(cmd, desc, cwd=None, env=None, tail=25, soft=False, timeout=None, logname=None,
       fix="Run this cell again. If it keeps failing, read the last log lines above."):
    """Run a command quietly (log file + heartbeat). On failure: short, readable error."""
    global LAST_LOG
    cmd = [str(c) for c in cmd]
    slug = re.sub(r"[^a-z0-9]+", "_", desc.lower()).strip("_")[:40]
    logf = LOGS / (logname or (slug + ".log"))
    LAST_LOG = logf
    print("⏳ " + desc + " ...", flush=True)
    t0 = time.time()
    p = None
    try:
        with open(logf, "wb") as fh:
            p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)
            beat = t0
            while p.poll() is None:
                time.sleep(1)
                now = time.time()
                if now - beat >= 30:
                    print("   … still working (%ds)" % (now - t0), flush=True)
                    beat = now
                if timeout and now - t0 > timeout:
                    p.kill()
                    break
            p.wait()
    except FileNotFoundError:
        if soft:
            print("   ⚠️ command not found: " + cmd[0])
            return False
        die("Command not found: " + cmd[0], "Run cell 2 again. If it is a system tool, run:  !apt-get install -y " + cmd[0])
    rc = p.returncode
    if rc == 0:
        print("✅ %s (%ds)" % (desc, time.time() - t0), flush=True)
        return True
    tail_txt = log_tail(logf, tail)
    if soft:
        print("   ⚠️ %s failed (exit %s). Last lines:\n%s" % (desc, rc, indent(tail_txt)))
        return False
    die("%s failed (exit code %s). Last lines of the log:\n%s" % (desc, rc, tail_txt), fix)

def gpu_info():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=30)
        if r.returncode != 0 or not r.stdout.strip():
            return None
        name, mem, drv = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
        return {"name": name, "vram_mb": int(float(mem)), "driver": drv}
    except Exception:
        return None

def ram_total_gb():
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal"):
                return int(line.split()[1]) / 1048576.0
    except Exception:
        pass
    return 0.0

def venv_env(extra=None):
    """Clean environment for the Python 3.13 venv (Colab's own PYTHONPATH / CUDA libs would poison it)."""
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "PYTHONSTARTUP")}
    ld = [p for p in env.get("LD_LIBRARY_PATH", "").split(":") if p and not p.startswith("/usr/local/cuda")]
    for d in ("/usr/lib64-nvidia", "/usr/local/nvidia/lib64"):
        if os.path.isdir(d) and d not in ld:
            ld.append(d)
    env["LD_LIBRARY_PATH"] = ":".join(ld)
    env["PATH"] = str(VENV / "bin") + ":" + env.get("PATH", "")
    env.update({
        "PYTHONUNBUFFERED": "1",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "TOKENIZERS_PARALLELISM": "false",
        "HF_HUB_DISABLE_TELEMETRY": "1",
        "LORALAB_SKIP_BROWSER": "1",
        "LORALAB_CLOUD": "1",
        "MPLBACKEND": "Agg",
    })
    if os.path.exists("/usr/lib64-nvidia/libcuda.so.1"):
        env["TRITON_LIBCUDA_PATH"] = "/usr/lib64-nvidia"
    if HF_TOKEN_FINAL:
        env["HF_TOKEN"] = HF_TOKEN_FINAL
        env["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN_FINAL
    if extra:
        env.update(extra)
    return env

HF_TOKEN_FINAL = (str(HF_TOKEN).strip() or get_secret("HF_TOKEN"))
NGROK_FINAL = (str(NGROK_TOKEN).strip() or get_secret("NGROK_AUTHTOKEN"))

def locate_trainer(tid):
    """Find the trainer's web server script (parsed from the repo's own Run_LoRAlab-*.sh)."""
    exports, script, entry = {}, None, None
    try:
        cfg = json.loads((APP / "GUI" / "launcher.json").read_text(encoding="utf-8"))
        entry = next((x for x in cfg.get("trainers", []) if x.get("id") == tid), None)
    except Exception:
        entry = None
    if entry:
        shp = APP / "code" / Path(entry["run"]).with_suffix(".sh")
        if shp.exists():
            txt = shp.read_text(encoding="utf-8", errors="ignore")
            m = re.search(r"scripts/(server_[A-Za-z0-9_]+\.py)", txt)
            if m and (APP / "scripts" / m.group(1)).exists():
                script = APP / "scripts" / m.group(1)
            for line in txt.splitlines():
                mm = re.match(r"\s*export\s+([A-Z_][A-Z0-9_]*)=(.*)$", line)
                if mm and "$" not in mm.group(2):
                    exports[mm.group(1)] = mm.group(2).strip().strip("\"'")
    if script is None and (APP / "scripts" / ("server_%s.py" % tid)).exists():
        script = APP / "scripts" / ("server_%s.py" % tid)
    if script is None:
        die("The trainer '%s' was not found in this version of the repository." % tid,
            "Pick another trainer in cell 1, keep REPO_REF = main, and re-run cell 4 to update the code.")
    return script, exports

# ─────────────────────────────────────────────────────────────────────────────
# Pre-flight: GPU / RAM / disk
# ─────────────────────────────────────────────────────────────────────────────
GPU = gpu_info()
if GPU is None:
    die("No GPU detected. This notebook needs the free Tesla T4.",
        "Menu: Runtime ▸ Change runtime type ▸ Hardware accelerator = T4 GPU ▸ Save. Then run all cells again "
        "(if T4 is unavailable, wait a few minutes: free GPUs are shared).")
RAM_GB = ram_total_gb()
DISK_FREE_GB = shutil.disk_usage(str(ROOT)).free / 1e9
if DISK_FREE_GB < 20:
    die("Only %.0f GB of disk space is free (need at least 20 GB)." % DISK_FREE_GB,
        "Run the '💽 Disk report' cell, or Runtime ▸ Disconnect and delete runtime, then start again.")

warnings = []
if "google.colab" not in sys.modules:
    warnings.append("Not running inside Google Colab: some features (Drive, downloads, keep-alive) will be skipped.")
if GPU["vram_mb"] < 14000:
    warnings.append("GPU has %.1f GB VRAM (a T4 has ~15 GB). Heavy trainers may run out of memory." % (GPU["vram_mb"] / 1024.0))
if RAM_GB and RAM_GB < 11:
    warnings.append("Only %.1f GB system RAM. Video trainers may not fit." % RAM_GB)

# ─────────────────────────────────────────────────────────────────────────────
# Optional swap (safety net for the 12 GB RAM limit; silently skipped if Colab blocks it)
# ─────────────────────────────────────────────────────────────────────────────
def add_swap(gb):
    if not gb or int(gb) <= 0:
        return "off"
    try:
        if len(Path("/proc/swaps").read_text().strip().splitlines()) > 1:
            return "already active"
    except Exception:
        pass
    swapfile = str(ROOT / "loralab.swap")
    try:
        subprocess.run(["fallocate", "-l", "%dG" % int(gb), swapfile], check=True, capture_output=True)
        os.chmod(swapfile, 0o600)
        subprocess.run(["mkswap", swapfile], check=True, capture_output=True)
        subprocess.run(["swapon", swapfile], check=True, capture_output=True)
        return "%d GB active" % int(gb)
    except Exception:
        try:
            os.remove(swapfile)
        except Exception:
            pass
        return "not available on this VM (skipped, no problem)"

SWAP_STATE = add_swap(SWAP_GB)

_warn_html = "".join('<div style="color:#fbbf24">⚠️ %s</div>' % w for w in warnings)
display(HTML(
    '<div style="font-family:system-ui,sans-serif;background:#0f172a;color:#e2e8f0;padding:14px 18px;border-radius:12px;border:1px solid #334155;line-height:1.7">'
    '<div style="font-size:16px;font-weight:700">✅ Runtime looks good</div>'
    '<div>🎮 GPU: <b>%s</b> · %.1f GB VRAM · driver %s</div>'
    '<div>🧠 System RAM: <b>%.1f GB</b> · 💽 Free disk: <b>%.0f GB</b> · 🔁 Swap: %s</div>'
    '<div>🧩 Trainer: <b>%s</b></div>%s</div>'
    % (GPU["name"], GPU["vram_mb"] / 1024.0, GPU["driver"], RAM_GB, DISK_FREE_GB, SWAP_STATE, TRAINER_ID, _warn_html)))

CORE_READY = True


In [ ]:
#@title 💾 3 · Mount Google Drive (optional — skipped unless enabled in Settings) { display-mode: "form" }
if "CORE_READY" not in globals():
    die("Cell 2 has not been run in this session.", "Run cell 1, then cell 2, then this cell (or Runtime ▸ Run all).")

DRIVE_ROOT = None
if not MOUNT_DRIVE:
    print("⏭️  Google Drive is OFF → everything stays on this temporary VM.")
    print("    Before the session ends use the '📤 Export LoRAs' cell or the in-app 'Download LoRA' button.")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        die("Could not mount Google Drive (%s: %s)." % (type(e).__name__, str(e)[:200]),
            "Run this cell again and click 'Connect to Google Drive' → choose your account → Allow. "
            "Or untick MOUNT_DRIVE in cell 1 to continue without Drive.")
    _my = Path("/content/drive/MyDrive")
    if not _my.exists():
        die("Drive is mounted but 'MyDrive' was not found.",
            "Run this cell again and make sure you allowed access for the same Google account.")
    _folder = re.sub(r"[^A-Za-z0-9 _.-]+", "_", str(DRIVE_FOLDER).strip()) or "LoRAlab"
    DRIVE_ROOT = _my / _folder
    try:
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        (DRIVE_ROOT / ".write_test").write_text("ok")
        (DRIVE_ROOT / ".write_test").unlink()
    except Exception as e:
        die("Drive folder is not writable (%s)." % str(e)[:160], "Free some Drive space (drive.google.com ▸ Storage) and run this cell again.")
    print("✅ Drive mounted. Persistent folder: %s" % DRIVE_ROOT)


⏭️  Google Drive is OFF → everything stays on this temporary VM.
    Before the session ends use the '📤 Export LoRAs' cell or the in-app 'Download LoRA' button.


In [ ]:
#@title 📦 4 · Install LoRAlab Trainer Studio (first run ≈ 6-10 min, re-runs are instant) { display-mode: "form" }
if "CORE_READY" not in globals():
    die("Cell 2 has not been run in this session.", "Run cells 1 → 2 → 3 first (or Runtime ▸ Run all).")
if "DRIVE_ROOT" not in globals():
    DRIVE_ROOT = None
INSTALL_DONE = False
T_INSTALL = time.time()

# ── 1) Trainer code ──────────────────────────────────────────────────────────
if not re.fullmatch(r"[A-Za-z0-9._/-]+", str(REPO_REF)):
    die("Invalid REPO_REF: %r" % REPO_REF, "Use a branch or tag name such as 'main'.")
if not (APP / ".git").exists():
    if APP.exists():
        shutil.rmtree(APP, ignore_errors=True)
    sh(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, APP], "Download trainer code from GitHub",
       fix="Check that GitHub is reachable (re-run the cell). If you changed REPO_REF, make sure that branch exists.")
else:
    sh(["git", "-C", APP, "pull", "--ff-only"], "Update trainer code", soft=True)

# ── 2) Google Drive persistence (symlinks) ───────────────────────────────────
def link_persist():
    if not DRIVE_ROOT:
        return
    names = ["Outputs", "settings", "dataset"]
    if str(PERSIST).startswith(("Medium", "Full")):
        names.append("Cache_Data")
    if str(PERSIST).startswith("Full"):
        names.append("Models")
    for n in names:
        target, link = DRIVE_ROOT / n, APP / n
        try:
            target.mkdir(parents=True, exist_ok=True)
            if link.is_symlink():
                if link.resolve() == target.resolve():
                    continue
                link.unlink()
            elif link.exists():
                if link.is_dir() and any(link.iterdir()):
                    shutil.copytree(str(link), str(target), dirs_exist_ok=True)
                shutil.rmtree(str(link)) if link.is_dir() else link.unlink()
            link.symlink_to(target, target_is_directory=True)
            print("🔗 %s → Drive/%s/%s" % (n, DRIVE_ROOT.name, n))
        except Exception as e:
            print("⚠️  Could not link %s to Drive (%s). It stays on the VM." % (n, str(e)[:120]))

link_persist()

# ── 3) Python 3.13 venv + PyTorch + dependencies ─────────────────────────────
UV = [sys.executable, "-m", "uv"]
UV_ENV = dict(os.environ, UV_LINK_MODE="copy", UV_NO_CACHE="1")
STATE_FILE = VENV / ".loralab_state.json"

def read_state():
    try:
        return json.loads(STATE_FILE.read_text())
    except Exception:
        return {}

def write_state(**kw):
    s = read_state()
    s.update(kw)
    STATE_FILE.write_text(json.dumps(s))

def venv_ok():
    if not PY.exists():
        return False
    try:
        r = subprocess.run([str(PY), "-c", "import sys;print(sys.version_info[:2]==(3,13))"], capture_output=True, text=True, timeout=60)
        return r.returncode == 0 and r.stdout.strip() == "True"
    except Exception:
        return False

def install_pkgs(desc, packages, attempts, force=False, fix="", soft=False):
    """attempts = [(tool, extra_args), ...]; the first one that works wins (uv first, pip as backup)."""
    for i, (tool, extra) in enumerate(attempts):
        if tool == "uv":
            cmd = UV + ["pip", "install", "--python", str(PY)] + (["--reinstall"] if force else []) + list(packages) + list(extra)
            env = UV_ENV
        else:
            cmd = [str(PY), "-m", "pip", "install", "--no-input", "--disable-pip-version-check"] + (["--force-reinstall"] if force else []) + list(packages) + list(extra)
            env = venv_env()
        label = desc if i == 0 else "%s (retry %d, %s)" % (desc, i, tool)
        slug = re.sub(r"[^a-z0-9]+", "_", desc.lower()).strip("_")[:24]
        if sh(cmd, label, env=env, soft=True, logname="install_%s_%d.log" % (slug, i), tail=12):
            return True
    if soft:
        return False
    die("%s could not be installed. Last output:\n%s" % (desc, log_tail(LAST_LOG, 25)),
        fix or "Run this cell again (it continues where it stopped). Usually it is a temporary network hiccup.")

TORCH_TEST = '''import sys, torch
if not torch.cuda.is_available():
    print("cuda_unavailable")
    sys.exit(2)
x = torch.randn(256, 256, device="cuda", dtype=torch.float16)
y = (x @ x).float().sum().item()
cap = torch.cuda.get_device_capability(0)
print("OK|%s|%s|%d.%d" % (torch.__version__, torch.cuda.get_device_name(0), cap[0], cap[1]))
'''

def verify_cuda():
    (ROOT / "loralab_torch_test.py").write_text(TORCH_TEST)
    try:
        r = subprocess.run([str(PY), str(ROOT / "loralab_torch_test.py")], env=venv_env(), capture_output=True, text=True, timeout=300, cwd=str(ROOT))
    except Exception:
        return None
    for line in r.stdout.splitlines():
        if line.startswith("OK|"):
            _, ver, dev, cap = line.split("|")
            return {"version": ver, "device": dev, "cap": cap}
    return None

def cuda_candidates(driver):
    try:
        major = int(str(driver).split(".")[0])
    except Exception:
        major = 0
    if major >= 580:
        return ["cu130", "cu128", "cu126"]
    if major >= 570:
        return ["cu128", "cu126"]
    return ["cu126", "cu128"]

def filtered_requirements():
    src = APP / "requirements.txt"
    if not src.exists():
        die("requirements.txt is missing from the downloaded repo.", "Delete /content/AcademiaSD_LoRAlab-TrainerStudio (run: !rm -rf /content/AcademiaSD_LoRAlab-TrainerStudio) and run this cell again.")
    out = []
    for line in src.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or s.startswith("-"):
            continue
        name = re.split(r"[ <>=!~;@\[]", s, maxsplit=1)[0].lower().replace("_", "-")
        if name in ("torch", "torchvision", "torchaudio"):
            continue          # PyTorch is installed separately with the right CUDA build
        out.append(s)
    return out

CHECK_SRC = '''import importlib, sys
mods = ["torch", "diffusers", "transformers", "peft", "accelerate", "bitsandbytes", "safetensors", "flask", "psutil"]
bad = []
for m in mods:
    try:
        importlib.import_module(m)
    except Exception as e:
        bad.append("%s -> %s: %s" % (m, type(e).__name__, str(e)[:200]))
print(chr(10).join(bad))
sys.exit(1 if bad else 0)
'''

def check_imports():
    (ROOT / "loralab_check.py").write_text(CHECK_SRC)
    try:
        r = subprocess.run([str(PY), str(ROOT / "loralab_check.py")], env=venv_env(), capture_output=True, text=True, timeout=400, cwd=str(ROOT))
        return r.returncode == 0, (r.stdout + r.stderr).strip()
    except Exception as e:
        return False, str(e)

# 3a) uv + venv
if not venv_ok():
    sh([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], "Install the 'uv' fast installer",
       fix="Check your connection and run the cell again.")
    if VENV.exists():
        shutil.rmtree(VENV, ignore_errors=True)
    sh(UV + ["venv", "--python", "3.13", "--seed", str(VENV)], "Create the Python 3.13 environment", env=UV_ENV,
       fix="uv downloads Python 3.13 from GitHub: make sure github.com is reachable, then run this cell again.")
    if not venv_ok():
        die("The Python 3.13 environment was created but does not start.", "Run:  !rm -rf /content/loralab_venv   then run this cell again.")
else:
    print("✅ Python 3.13 environment already exists.")

# 3b) PyTorch
state = read_state()
torch_info = verify_cuda() if state.get("torch_tag") else None
if torch_info:
    print("✅ PyTorch %s already working on %s." % (torch_info["version"], torch_info["device"]))
else:
    chosen = None
    for n, tag in enumerate(cuda_candidates(GPU["driver"])):
        idx = "https://download.pytorch.org/whl/" + tag
        ok = install_pkgs("PyTorch (%s)" % tag, ["torch", "torchvision", "torchaudio"],
                          [("uv", ["--index-url", idx]), ("uv", ["--extra-index-url", idx]), ("pip", ["--index-url", idx])],
                          force=(n > 0), soft=True)
        torch_info = verify_cuda() if ok else None
        if torch_info:
            chosen = tag
            break
        print("⚠️  PyTorch %s did not work on this GPU/driver. Trying the next build ..." % tag)
    if not chosen:
        die("PyTorch could not be set up with a working GPU (driver %s)." % GPU["driver"],
            "Runtime ▸ Disconnect and delete runtime, reconnect with a T4 GPU, and run the cells again. "
            "If it repeats, run '!rm -rf /content/loralab_venv' and re-run this cell.")
    write_state(torch_tag=chosen)
    print("✅ PyTorch %s ready (%s)." % (torch_info["version"], chosen))

# 3c) Trainer dependencies (diffusers, transformers, peft, bitsandbytes ...)
req = filtered_requirements()
req_hash = hashlib.sha1("\n".join(req).encode()).hexdigest()[:12]
state = read_state()
imports_ok = False
if state.get("req_hash") == req_hash:
    imports_ok, _msg = check_imports()
if imports_ok:
    print("✅ Trainer dependencies already installed.")
else:
    (ROOT / "loralab_req.txt").write_text("\n".join(req) + "\n")
    _code = "from importlib.metadata import version as v\nfor n in ('torch','torchvision','torchaudio'):\n    print(n + '==' + v(n).split('+')[0])"
    _r = subprocess.run([str(PY), "-c", _code], capture_output=True, text=True, env=venv_env())
    (ROOT / "loralab_constraints.txt").write_text(_r.stdout)
    install_pkgs("Trainer dependencies (diffusers, transformers, peft, bitsandbytes ...)",
                 ["-r", str(ROOT / "loralab_req.txt"), "-c", str(ROOT / "loralab_constraints.txt")],
                 [("uv", []), ("pip", [])],
                 fix="Run this cell again; it resumes. If one package keeps failing, the log lines above name it.")
    ok, msg = check_imports()
    if not ok:
        write_state(req_hash="")
        die("Some packages were installed but fail to import:\n" + msg,
            "Run this cell again (it repairs the environment). If it repeats, run '!rm -rf /content/loralab_venv' and re-run this cell.")
    write_state(req_hash=req_hash)
    print("✅ Dependencies installed and verified.")

# ── 4) ffmpeg (needed for video clips) ───────────────────────────────────────
if not shutil.which("ffmpeg"):
    sh(["apt-get", "install", "-y", "-qq", "ffmpeg"], "Install ffmpeg", soft=True)

# ── 5) Run wrapper (adds login support behind tunnels) ───────────────────────
WRAPPER_SRC = '''# Auto-generated by the Colab notebook
import os, sys, runpy
from pathlib import Path
script = Path(sys.argv[1]).resolve()
os.chdir(str(script.parent.parent))
sys.path.insert(0, str(script.parent))
sys.argv = [str(script)]
if os.environ.get("LL_PROXYFIX", "1") == "1":
    # Tunnels connect from 127.0.0.1; trusting X-Forwarded-For lets the app's own login protect public traffic.
    try:
        import flask
        from werkzeug.middleware.proxy_fix import ProxyFix
        _orig = flask.Flask.__init__
        def _patched(self, *a, **kw):
            _orig(self, *a, **kw)
            self.wsgi_app = ProxyFix(self.wsgi_app, x_for=1, x_proto=1)
        flask.Flask.__init__ = _patched
    except Exception as e:
        print("[colab] ProxyFix not applied:", e, flush=True)
runpy.run_path(str(script), run_name="__main__")
'''
WRAPPER.write_text(WRAPPER_SRC)

# ── 6) Login + token settings ────────────────────────────────────────────────
WEB_USER_FINAL = (str(WEB_USER).strip().replace(":", "") or "loralab")
WEB_PW_FINAL = (str(WEB_PASSWORD).strip() or secrets.token_urlsafe(9))
pw_hash = ""
if REQUIRE_PASSWORD:
    _r = subprocess.run([str(PY), "-c", "import os;from werkzeug.security import generate_password_hash as g;print(g(os.environ['LL_PW']))"],
                        capture_output=True, text=True, env=venv_env({"LL_PW": WEB_PW_FINAL}))
    if _r.returncode != 0 or not _r.stdout.strip():
        die("Could not create the login hash: " + (_r.stderr or "")[-300:], "Run this cell again.")
    pw_hash = _r.stdout.strip().splitlines()[-1]

(APP / "settings").mkdir(exist_ok=True)
(APP / "settings" / "network.json").write_text(json.dumps({
    "listen": False, "launcher_port": 4990, "trainer_port": PORT,
    "user": WEB_USER_FINAL if REQUIRE_PASSWORD else "", "password_hash": pw_hash,
    "external_auth": False, "trainer_url": ""}, indent=2))
if HF_TOKEN_FINAL and not DRIVE_ROOT:
    # Not written to Drive on purpose; with Drive the token is passed through the environment only.
    (APP / "settings" / "HF_token.json").write_text(json.dumps({"token": HF_TOKEN_FINAL}))

# ── 7) Final check ───────────────────────────────────────────────────────────
_script, _ = locate_trainer(TRAINER_ID)
INSTALL_DONE = True
print("\n✅ Installation complete in %s. Trainer server: %s" % (hms(time.time() - T_INSTALL), _script.name))
print("   Next: run cell 5 (🚀 Launch).")


⏳ Download trainer code from GitHub ...
✅ Download trainer code from GitHub (4s)
⏳ Install the 'uv' fast installer ...
✅ Install the 'uv' fast installer (4s)
⏳ Create the Python 3.13 environment ...
✅ Create the Python 3.13 environment (1s)
⏳ PyTorch (cu130) ...
   … still working (30s)
   … still working (60s)
✅ PyTorch (cu130) (88s)
✅ PyTorch 2.14.1+cu130 ready (cu130).
⏳ Trainer dependencies (diffusers, transformers, peft, bitsandbytes ...) ...
✅ Trainer dependencies (diffusers, transformers, peft, bitsandbytes ...) (16s)
✅ Dependencies installed and verified.

✅ Installation complete in 0:02:20. Trainer server: server_krea2.py
   Next: run cell 5 (🚀 Launch).


In [ ]:
#@title 🚀 5 · Launch: trainer + public tunnels + keep-alive  (leave this cell running!) { display-mode: "form" }
if "INSTALL_DONE" not in globals() or not INSTALL_DONE:
    die("The install step has not finished.", "Run cell 4 (📦 Install) and wait for '✅ Installation complete', then run this cell.")

SERVER = globals().get("SERVER") or {"proc": None, "restarts": 0}
TUN = {}
MAX_RESTARTS = 3

def event(msg):
    print("[%s] %s" % (time.strftime("%H:%M:%S"), msg), flush=True)

# ── Trainer server ───────────────────────────────────────────────────────────
def start_server():
    script, exports = locate_trainer(TRAINER_ID)
    logf = open(LOGS / "server.log", "ab")
    logf.write(("\n===== start %s =====\n" % time.ctime()).encode())
    logf.flush()
    extra = dict(exports)
    extra["LL_PROXYFIX"] = "1" if REQUIRE_PASSWORD else "0"
    proc = subprocess.Popen([str(PY), "-u", str(WRAPPER), str(script)], cwd=str(APP), env=venv_env(extra),
                            stdout=logf, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
    logf.close()
    SERVER["proc"] = proc
    return proc

def wait_for_server(proc, timeout=300):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc is not None and proc.poll() is not None:
            die("The trainer server crashed while starting (exit code %s). Last log lines:\n%s" % (proc.returncode, log_tail(LOGS / "server.log", 30)),
                "Read the last lines above. Typical fixes: re-run cell 4 (repairs packages), pick another trainer in cell 1, or "
                "Runtime ▸ Disconnect and delete runtime if the VM is in a bad state.")
        if port_open():
            st, _ = http_get("http://127.0.0.1:%d/api/status" % PORT, timeout=5)
            if st == 200:
                return True
        time.sleep(2)
    die("The trainer server did not answer within %d s." % timeout,
        "Run the '📜 Show logs' cell (Trainer server) to see why, then run this cell again.")

def trainer_state():
    st, body = http_get("http://127.0.0.1:%d/api/status" % PORT, timeout=5)
    if st != 200:
        return None
    try:
        return json.loads(body.decode("utf-8", "ignore"))
    except Exception:
        return None

# ── Tunnels ──────────────────────────────────────────────────────────────────
def _scan_url(logpath, regex, exclude_prefix=()):
    try:
        txt = Path(logpath).read_text(errors="ignore")
    except Exception:
        return None
    for m in re.finditer(regex, txt):
        u = m.group(0)
        if not u.startswith(exclude_prefix):
            return u
    return None

def _spawn(name, cmd, regex, timeout=50, exclude_prefix=()):
    logp = LOGS / ("tunnel_%s.log" % name)
    fh = open(logp, "wb")
    proc = subprocess.Popen([str(c) for c in cmd], stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
    fh.close()
    t0 = time.time()
    while time.time() - t0 < timeout:
        url = _scan_url(logp, regex, exclude_prefix)
        if url:
            return proc, url
        if proc.poll() is not None:
            break
        time.sleep(1)
    _kill(proc)
    return None, None

CLOUDFLARED = ROOT / "cloudflared"

def ensure_cloudflared():
    if CLOUDFLARED.exists() and os.access(str(CLOUDFLARED), os.X_OK):
        return True
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    for _ in range(3):
        try:
            tmp = str(CLOUDFLARED) + ".part"
            urllib.request.urlretrieve(url, tmp)
            os.chmod(tmp, 0o755)
            os.replace(tmp, str(CLOUDFLARED))
            return True
        except Exception:
            time.sleep(2)
    return False

def start_cloudflare():
    if not ensure_cloudflared():
        return None, "could not download cloudflared from github.com"
    for _ in range(3):
        proc, url = _spawn("cloudflare", [CLOUDFLARED, "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", "http://127.0.0.1:%d" % PORT],
                           r"https://[a-z0-9-]+\.trycloudflare\.com", 50, ("https://api.",))
        if url:
            return {"kind": "cloudflare", "url": url, "proc": proc}, ""
        time.sleep(3)
    return None, "Cloudflare gave no URL:\n" + indent(log_tail(LOGS / "tunnel_cloudflare.log", 6))

SSH_OPTS = ["-T", "-o", "StrictHostKeyChecking=no", "-o", "UserKnownHostsFile=/dev/null", "-o", "ServerAliveInterval=20",
            "-o", "ServerAliveCountMax=3", "-o", "ExitOnForwardFailure=yes", "-o", "LogLevel=ERROR"]

def ensure_ssh():
    if shutil.which("ssh"):
        return True
    subprocess.run(["apt-get", "install", "-y", "-qq", "openssh-client"], capture_output=True)
    return bool(shutil.which("ssh"))

def start_ngrok():
    if not NGROK_FINAL:
        return None, "no ngrok token"
    try:
        try:
            import pyngrok
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], capture_output=True)
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_FINAL)
        for old in ngrok.get_tunnels():
            ngrok.disconnect(old.public_url)
        tun = ngrok.connect("127.0.0.1:%d" % PORT, "http")
        return {"kind": "ngrok", "url": tun.public_url.replace("http://", "https://"), "proc": None}, ""
    except Exception as e:
        return None, "ngrok: " + str(e)[:300]

def start_localhostrun():
    proc, url = _spawn("localhostrun", ["ssh"] + SSH_OPTS + ["-R", "80:127.0.0.1:%d" % PORT, "nokey@localhost.run"], r"https://[a-z0-9-]+\.lhr\.life", 45)
    if url:
        return {"kind": "localhostrun", "url": url, "proc": proc}, ""
    return None, "localhost.run gave no URL"

def start_pinggy():
    proc, url = _spawn("pinggy", ["ssh"] + SSH_OPTS + ["-p", "443", "-R", "0:127.0.0.1:%d" % PORT, "free.pinggy.io"], r"https://[a-z0-9.-]+\.pinggy\.(?:link|online)", 45)
    if url:
        return {"kind": "pinggy", "url": url, "proc": proc, "note": "(free plan: 60 min per link)"}, ""
    return None, "Pinggy gave no URL"

def start_fallback():
    errs = []
    if NGROK_FINAL:
        t, e = start_ngrok()
        if t:
            return t, ""
        errs.append(e)
    if ensure_ssh():
        for fn in (start_localhostrun, start_pinggy):
            t, e = fn()
            if t:
                return t, ""
            errs.append(e)
    else:
        errs.append("ssh client unavailable")
    return None, "; ".join(errs)

def stop_tunnels():
    for t in list(TUN.values()):
        _kill(t.get("proc"))
    TUN.clear()
    try:
        from pyngrok import ngrok
        ngrok.kill()
    except Exception:
        pass
    for pat in ("cloudflared", "localhost.run", "free.pinggy.io"):
        try:
            subprocess.run(["pkill", "-f", pat], capture_output=True)
        except Exception:
            pass

def verify_tunnel(url, secs=60):
    t0, last = time.time(), None
    while time.time() - t0 < secs:
        st, _ = http_get(url + "/api/status", timeout=10)
        last = st
        if st in (200, 401):
            return st
        time.sleep(3)
    return last

LABELS = {"cloudflare": "Cloudflare (primary)", "ngrok": "ngrok (fallback)", "localhostrun": "localhost.run (fallback)", "pinggy": "Pinggy (fallback)"}

def links_html():
    cards = ""
    for name, t in TUN.items():
        cards += ('<div style="margin:8px 0;padding:10px 12px;background:#111c33;border-radius:10px">'
                  '<div style="color:#94a3b8;font-size:12px">%s %s</div>'
                  '<a href="%s" target="_blank" style="color:#7dd3fc;font-size:17px;font-weight:600;word-break:break-all">%s</a></div>'
                  % (LABELS.get(t["kind"], t["kind"]), t.get("note", ""), t["url"], t["url"]))
    if not cards:
        cards = '<div style="color:#fca5a5;margin:8px 0">No tunnel is up right now. Retrying automatically ...</div>'
    if REQUIRE_PASSWORD:
        cred = ('<div style="margin-top:6px">🔒 Login → user <b style="color:#fde68a">%s</b> · password <b style="color:#fde68a">%s</b></div>'
                % (WEB_USER_FINAL, WEB_PW_FINAL))
    else:
        cred = '<div style="color:#fbbf24;margin-top:6px">⚠️ Password protection is OFF: anyone with the link can use your trainer.</div>'
    return ('<div style="font-family:system-ui,sans-serif;background:#0f172a;color:#e2e8f0;padding:16px 18px;border-radius:14px;border:1px solid #334155">'
            '<div style="font-size:18px;font-weight:700;margin-bottom:4px">🌐 Your trainer is online</div>%s%s'
            '<div style="color:#94a3b8;font-size:12px;margin-top:8px">Open a link in a new tab. Cloudflare is the primary one; '
            'if the live console or previews ever lag, use the fallback link.</div></div>' % (cards, cred))

# ── Keep-alive ───────────────────────────────────────────────────────────────
def silent_wav_b64():
    buf = io.BytesIO()
    w = wave.open(buf, "wb")
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(8000)
    w.writeframes(b"".join(struct.pack("<h", int(6 * math.sin(2 * math.pi * 30 * i / 8000.0))) for i in range(16000)))
    w.close()
    return base64.b64encode(buf.getvalue()).decode()

KA_JS = """
(function () {
  if (window.__loralab_ka) { return; }
  window.__loralab_ka = true;
  function tryClick() {
    try {
      var docs = [document];
      try { docs.push(window.parent.document); } catch (e) {}
      try { docs.push(window.top.document); } catch (e) {}
      for (var i = 0; i < docs.length; i++) {
        var b = docs[i].querySelector('colab-connect-button');
        if (b && b.shadowRoot) { var x = b.shadowRoot.querySelector('#connect'); if (x) { x.click(); return true; } }
        var t = docs[i].querySelector('colab-toolbar-button#connect');
        if (t) { t.click(); return true; }
      }
    } catch (e) {}
    return false;
  }
  function beat() {
    tryClick();
    try { google.colab.kernel.invokeFunction('loralab.ping', [Date.now()], {}); } catch (e) {}
  }
  setInterval(beat, 45000);
  beat();
})();
"""

def start_keepalive():
    if KEEPALIVE_JS:
        try:
            from google.colab import output as _out
            _out.register_callback("loralab.ping", lambda *a: None)
            display(Javascript(KA_JS))
        except Exception:
            print("ℹ️ Browser heartbeat unavailable here (not Colab?). Skipping.")
    if KEEPALIVE_AUDIO:
        display(HTML('<div style="font-family:system-ui;font-size:12px;color:#94a3b8">🔈 keep-alive audio (inaudible loop). If it is not playing, press ▶ once: '
                     '<audio controls autoplay loop style="height:28px;vertical-align:middle" src="data:audio/wav;base64,%s"></audio></div>' % silent_wav_b64()))

# ── Live status ──────────────────────────────────────────────────────────────
def gpu_metrics():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=10)
        u, mu, mt, tp = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
        return "%s%% · %.1f/%.1f GB · %s°C" % (u, int(mu) / 1024.0, int(mt) / 1024.0, tp)
    except Exception:
        return "n/a"

def ram_metrics():
    try:
        import psutil
        m = psutil.virtual_memory()
        return "%.1f/%.1f GB" % (m.used / 2**30, m.total / 2**30)
    except Exception:
        return "n/a"

def render_status(t_start, tr, beat):
    if tr is None:
        train_txt = "trainer: <span style='color:#fca5a5'>not answering</span>"
    elif tr.get("running"):
        p = tr.get("progress") or {}
        prog = (" · step %s/%s" % (p.get("step"), p.get("total"))) if p.get("step") is not None else ""
        train_txt = "<span style='color:#fbbf24'>⚙ %s running%s</span>" % (tr.get("script"), prog)
    else:
        train_txt = "idle (waiting in the web UI)"
    tun_txt = ", ".join(LABELS.get(t["kind"], t["kind"]).split(" ")[0] for t in TUN.values()) or "<span style='color:#fca5a5'>none</span>"
    free = shutil.disk_usage(str(ROOT)).free / 1e9
    return ('<div style="font-family:ui-monospace,monospace;font-size:13px;line-height:1.7;background:#0b1220;color:#cbd5e1;padding:12px 14px;border-radius:10px;border:1px solid #1e293b">'
            '<b style="color:#34d399">● RUNNING</b> &nbsp;uptime %s &nbsp;|&nbsp; %s &nbsp;|&nbsp; %s<br>'
            'GPU %s &nbsp;|&nbsp; RAM %s &nbsp;|&nbsp; disk free %.0f GB<br>'
            'tunnels: %s &nbsp;|&nbsp; keep-alive %s &nbsp;|&nbsp; updated %s</div>'
            % (hms(time.time() - t_start), TRAINER_ID, train_txt, gpu_metrics(), ram_metrics(), free, tun_txt, beat, time.strftime("%H:%M:%S")))

# ═════════════════════════════ MAIN ═════════════════════════════
print("🚀 Starting %s ...\n" % TRAINER_ID)

# 1) server (never kill a trainer that is already running: reuse it)
if port_open():
    event("Trainer already running on port %d: reusing it (training is not interrupted)." % PORT)
else:
    SERVER["restarts"] = 0
    proc = start_server()
    event("Trainer server starting (first start can take ~1 min) ...")
    wait_for_server(proc)
    event("✅ Trainer server is up.")

# 2) tunnels
stop_tunnels()
mode = str(TUNNELS)
tunnel_errors = []
starters = []
if not mode.startswith("Fallback"):
    starters.append(("cloudflare", start_cloudflare))
if not mode.startswith("Cloudflare"):
    starters.append(("fallback", start_fallback))
for key, fn in starters:
    event("Opening tunnel: %s ..." % key)
    t, err = fn()
    if t:
        TUN[key] = t
        t["starter"] = fn
        t["fail"] = 0
        st = verify_tunnel(t["url"], 60)
        if st == 401 or (st == 200 and not REQUIRE_PASSWORD):
            event("✅ %s reachable%s" % (t["url"], " (login required)" if st == 401 else ""))
        elif st == 200:
            event("⚠️ %s is reachable WITHOUT a login (this tunnel hides the visitor address). Do not share the link." % t["url"])
        else:
            event("ℹ️ Could not verify %s from inside Colab (status %s). It often still works from your browser: try it." % (t["url"], st))
    else:
        tunnel_errors.append("%s: %s" % (key, err))
        event("⚠️ %s tunnel failed: %s" % (key, err))

if not TUN:
    die("No public tunnel could be started.\n" + "\n".join(tunnel_errors),
        "Run this cell again. If Cloudflare keeps failing set Tunnels = 'Fallback only' and add a free ngrok token (dashboard.ngrok.com) in cell 1.")

links_disp = display(HTML(links_html()), display_id=True)
start_keepalive()
status_disp = display(HTML("starting ..."), display_id=True)
print("\n💡 Keep this cell running. Stopping it closes the tunnels only; a running training continues and re-running this cell reconnects.\n")

# 3) monitor loop
t_start = time.time()
last_tunnel_check = time.time()
was_running = False
down_count = 0
try:
    while True:
        now = time.time()

        # server health
        proc = SERVER.get("proc")
        dead = (proc is not None and proc.poll() is not None) or not port_open()
        down_count = down_count + 1 if dead else 0
        if down_count >= 3:
            if AUTO_RESTART_SERVER and SERVER["restarts"] < MAX_RESTARTS:
                SERVER["restarts"] += 1
                event("⚠️ Trainer server stopped. Restarting (%d/%d) ..." % (SERVER["restarts"], MAX_RESTARTS))
                _kill(proc)
                time.sleep(2)
                np_ = start_server()
                wait_for_server(np_)
                event("✅ Trainer server is back.")
                down_count = 0
            else:
                die("The trainer server stopped and was not restarted.\n" + log_tail(LOGS / "server.log", 25),
                    "Run cell 5 again. Check '📜 Show logs' ▸ Trainer server for the reason (often out-of-memory).")

        # tunnel health (every 60 s)
        if now - last_tunnel_check >= 60:
            last_tunnel_check = now
            changed = False
            for key in list(TUN.keys()):
                t = TUN[key]
                alive = True
                if t["kind"] == "ngrok":
                    try:
                        from pyngrok import ngrok
                        alive = len(ngrok.get_tunnels()) > 0
                    except Exception:
                        alive = False
                elif t.get("proc") is not None:
                    alive = t["proc"].poll() is None
                st, _ = http_get(t["url"] + "/api/status", timeout=10)
                t["fail"] = 0 if st in (200, 401) else t.get("fail", 0) + 1
                if (not alive) or t["fail"] >= 3:
                    event("⚠️ %s tunnel lost. Opening a new one ..." % t["kind"])
                    _kill(t.get("proc"))
                    fn = t["starter"]
                    nt, err = fn()
                    if nt:
                        nt["starter"] = fn
                        nt["fail"] = 0
                        TUN[key] = nt
                        event("✅ New %s link: %s" % (nt["kind"], nt["url"]))
                    else:
                        event("❌ Could not reopen %s tunnel (%s). Will retry." % (key, err))
                    changed = True
            if changed:
                links_disp.update(HTML(links_html()))

        # training state
        tr = trainer_state()
        running_now = bool(tr and tr.get("running"))
        if was_running and not running_now and tr is not None:
            last = tr.get("last") or {}
            if last.get("code") in (0, None):
                event("✅ Job '%s' finished." % last.get("script"))
            else:
                event("⚠️ Job '%s' ended with code %s. Open the console in the web UI (or '📜 Show logs')." % (last.get("script"), last.get("code")))
        was_running = running_now

        beat = "♥" if int(now) % 20 < 10 else "♡"
        status_disp.update(HTML(render_status(t_start, tr, beat)))
        time.sleep(10)
except KeyboardInterrupt:
    stop_tunnels()
    print("\n⏸️  Monitor stopped and tunnels closed.")
    if port_open():
        print("   The trainer is STILL RUNNING on this VM (any training continues).")
        print("   → Re-run this cell to get new links, or run '⏹️ Stop everything' to shut it all down.")


🚀 Starting krea2 ...

[18:53:24] Trainer server starting (first start can take ~1 min) ...
[18:53:26] ✅ Trainer server is up.
[18:53:26] Opening tunnel: cloudflare ...
[18:54:04] ✅ https://theatre-fixtures-herbal-gold.trycloudflare.com reachable
[18:54:04] Opening tunnel: fallback ...


[18:54:15] ✅ https://3ac087598ba3de.lhr.life reachable


<IPython.core.display.Javascript object>


💡 Keep this cell running. Stopping it closes the tunnels only; a running training continues and re-running this cell reconnects.

[19:00:25] ⚠️ localhostrun tunnel lost. Opening a new one ...
[19:00:28] ✅ New localhostrun link: https://ebc072d9a688fe.lhr.life
[19:03:03] ⚠️ Job 'precache' ended with code 1. Open the console in the web UI (or '📜 Show logs').


---
## 🧰 Utilities
Run these from a **second tab of the notebook** or after stopping cell 5. They never touch a running training unless you use ⏹️ Stop.


In [ ]:
#@title 📜 Show logs { display-mode: "form" }
LOG_WHICH = "Trainer server" #@param ["Trainer server", "Training console (last run)", "Cloudflare tunnel", "Fallback tunnels", "Install"]
LOG_LINES = 80 #@param {type:"integer"}

_root, _logs = Path("/content"), Path("/content/loralab_logs")
_tid = globals().get("TRAINER_ID", "")
_map = {
    "Trainer server": [_logs / "server.log"],
    "Training console (last run)": sorted((_root / "AcademiaSD_LoRAlab-TrainerStudio" / "settings").glob("trainer_console_%s*.log" % _tid)) if _tid else [],
    "Cloudflare tunnel": [_logs / "tunnel_cloudflare.log"],
    "Fallback tunnels": [_logs / "tunnel_localhostrun.log", _logs / "tunnel_pinggy.log"],
    "Install": sorted(_logs.glob("install_*.log")) + sorted(_logs.glob("trainer_dependencies*.log")),
}
_shown = False
for _p in _map.get(LOG_WHICH, []):
    if Path(_p).exists():
        _shown = True
        _txt = Path(_p).read_text(errors="ignore")
        _txt = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", _txt).replace("\r", "\n")
        _lines = [l for l in _txt.splitlines() if l.strip()]
        print("═" * 20, Path(_p).name, "(last %d lines)" % LOG_LINES, "═" * 20)
        print("\n".join(_lines[-int(LOG_LINES):]))
if not _shown:
    print("ℹ️ No log for '%s' yet. It appears after that step has run." % LOG_WHICH)


In [ ]:
#@title 📤 Export LoRAs (download to your computer and/or copy to Drive) { display-mode: "form" }
EXPORT_WHAT = "Final LoRAs only" #@param ["Final LoRAs only", "All checkpoints"]
EXPORT_TO = "Download to my computer" #@param ["Download to my computer", "Copy to Google Drive", "Both"]

import zipfile
_out = Path("/content/AcademiaSD_LoRAlab-TrainerStudio/Outputs")
if not _out.exists():
    raise LoRALabError("No Outputs folder yet: nothing to export.\n\n👉 HOW TO FIX: Train a LoRA first (or run cells 4-5).") if "LoRALabError" in globals() else SystemExit("No Outputs folder yet.")

def _step(f):
    m = re.search(r"step_?(\d+)", f.name, re.I)
    return int(m.group(1)) if m else -1

_files = [f for f in sorted(_out.rglob("*.safetensors")) if "resume_checkpoint" not in str(f)]
if EXPORT_WHAT.startswith("Final"):
    _by, _sel = {}, []
    for f in _files:
        _by.setdefault(f.parent, []).append(f)
    for d, fs in _by.items():
        finals = [f for f in fs if "final" in f.name.lower()]
        _sel += finals if finals else [max(fs, key=_step)]
    _files = _sel
if not _files:
    raise LoRALabError("No .safetensors LoRA files found in Outputs.\n\n👉 HOW TO FIX: Wait until a checkpoint was saved (check the web UI console), then run this cell again.") if "LoRALabError" in globals() else SystemExit("No LoRA files found.")

print("Exporting %d file(s):" % len(_files))
for f in _files:
    print("  • %s  (%.0f MB)" % (f.relative_to(_out), f.stat().st_size / 1e6))

if EXPORT_TO in ("Download to my computer", "Both"):
    _zip = "/content/LoRAs_export.zip"
    with zipfile.ZipFile(_zip, "w", zipfile.ZIP_STORED) as z:
        for f in _files:
            z.write(str(f), str(f.relative_to(_out)))
    try:
        from google.colab import files as _cf
        _cf.download(_zip)
        print("⬇️  Download started (allow pop-ups if your browser blocks it).")
    except Exception as e:
        print("ℹ️ Automatic download unavailable (%s). Use the Files panel (📁) to download %s" % (type(e).__name__, _zip))

if EXPORT_TO in ("Copy to Google Drive", "Both"):
    try:
        if not Path("/content/drive/MyDrive").exists():
            from google.colab import drive as _drive
            _drive.mount("/content/drive")
        _dst = Path("/content/drive/MyDrive") / (globals().get("DRIVE_FOLDER", "LoRAlab") or "LoRAlab") / "exports"
        _dst.mkdir(parents=True, exist_ok=True)
        for f in _files:
            shutil.copyfile(str(f), str(_dst / ("%s__%s" % (f.parent.name, f.name))))
        print("✅ Copied to Drive: %s" % _dst)
    except Exception as e:
        print("⚠️ Could not copy to Drive (%s: %s). Run the cell again and allow Drive access." % (type(e).__name__, str(e)[:150]))


In [ ]:
#@title 💽 Disk report (and optional cache cleanup) { display-mode: "form" }
CLEAN_PIP_CACHES = False #@param {type:"boolean"}

_free = shutil.disk_usage("/content").free / 1e9
print("Free disk on the VM: %.1f GB\n" % _free)
for _p in ["/content/AcademiaSD_LoRAlab-TrainerStudio/Models", "/content/AcademiaSD_LoRAlab-TrainerStudio/Cache_Data",
           "/content/AcademiaSD_LoRAlab-TrainerStudio/Outputs", "/content/AcademiaSD_LoRAlab-TrainerStudio/dataset",
           "/content/loralab_venv", os.path.expanduser("~/.cache/huggingface")]:
    if os.path.exists(_p):
        try:
            _r = subprocess.run(["du", "-shL", _p], capture_output=True, text=True, timeout=120)
            print("%8s  %s" % (_r.stdout.split()[0] if _r.stdout else "?", _p))
        except Exception:
            print("%8s  %s" % ("?", _p))
if CLEAN_PIP_CACHES:
    for _c in ("~/.cache/pip", "~/.cache/uv"):
        shutil.rmtree(os.path.expanduser(_c), ignore_errors=True)
    subprocess.run(["apt-get", "clean"], capture_output=True)
    print("\n🧹 pip/uv/apt caches removed. Free now: %.1f GB" % (shutil.disk_usage("/content").free / 1e9))
print("\nTip: models are re-downloaded when missing, so deleting Models/ in the file browser is safe when you need space.")


In [ ]:
#@title ⏹️ Stop everything (trainer, training and tunnels) { display-mode: "form" }
_stopped = []
try:
    _p = globals().get("SERVER", {}).get("proc")
    if _p is not None:
        os.killpg(os.getpgid(_p.pid), signal.SIGTERM)
        _stopped.append("trainer server")
except Exception:
    pass
for _pat, _label in (("AcademiaSD_LoRAlab-TrainerStudio/scripts/", "trainer / training processes"), ("cloudflared", "Cloudflare tunnel"),
                     ("localhost.run", "localhost.run tunnel"), ("free.pinggy.io", "Pinggy tunnel")):
    try:
        _r = subprocess.run(["pkill", "-f", _pat], capture_output=True)
        if _r.returncode == 0:
            _stopped.append(_label)
    except Exception:
        pass
try:
    from pyngrok import ngrok as _ng
    _ng.kill()
    _stopped.append("ngrok")
except Exception:
    pass
if "TUN" in globals():
    TUN.clear()
print("⏹️  Stopped: " + (", ".join(_stopped) if _stopped else "nothing was running"))
print("To start again run cell 5. GPU memory is released when the processes exit.")


## 🩺 Troubleshooting

| Message / symptom | Fix |
|---|---|
| **No GPU detected** | Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save, then Run all. |
| **Disk space < 20 GB** | Runtime ▸ Disconnect and delete runtime, reconnect, run again. Use 💽 Disk report. |
| **Python 3.13 / uv download failed** | Needs github.com. Re-run cell 4 (it resumes). |
| **PyTorch could not be set up** | `!rm -rf /content/loralab_venv` then re-run cell 4. |
| **Packages fail to import** | Re-run cell 4 (auto-repair). If it repeats: delete the venv as above. |
| **Server crashed while starting** | 📜 Show logs ▸ *Trainer server*. Try another trainer, or re-run cell 4. |
| **Cloudflare gave no URL** | Re-run cell 5, or use *Fallback only* and add a free ngrok token. |
| **Console / previews freeze in the web UI** | The live console uses streaming (SSE). Open the **fallback link** instead of Cloudflare. |
| **Out of memory (CUDA)** | Lower resolution to 512, rank 8-16, batch 1; close preview-heavy options; try a ✅ model. |
| **Killed / RAM crash during model load** | Enable swap (cell 1, slider), avoid video trainers on free RAM, restart the runtime and re-launch. |
| **Session disconnected** | Runtime ▸ Run all again. With Drive enabled your outputs, dataset and settings are still there; trainers resume from the last checkpoint. |
| **Drive mount failed** | Re-run cell 3 and allow access, or untick *MOUNT_DRIVE*. |

### 💓 About keep-alive
The notebook keeps a cell executing with live output, sends a browser heartbeat every 45 s, tries to press Colab's *Connect* button, plays an inaudible audio loop and restarts the server if it dies.
It **cannot** override Google's limits (max ~12 h, and free GPUs can be reclaimed). Keep the tab open, and for long runs use Drive or export checkpoints regularly.

If you also want a manual heartbeat, paste this in your browser's developer console (F12 ▸ Console) on the Colab tab:
`setInterval(function(){document.querySelector("colab-connect-button")?.shadowRoot?.getElementById("connect")?.click()},60000)`

### 🔒 About the public link
Anyone who has the link **and** the password can use your trainer. Keep REQUIRE_PASSWORD on and do not post the link publicly.
